# 🚀 ComfyUI + MiniMax-H3 в Google Colab (A100 GPU Edition)

Готовый шаблон «всё в одном» для запуска мощной модели генерации видео и звука **MiniMax-H3 (Hailuo)** на видеокарте **A100 (40 ГБ VRAM)** с использованием **Colab Compute Units (CU)** из подписки Google AI Pro.

> 📖 **Подробная статья и разбор (Zenn, на японском)**: [Как с подпиской Google AI Pro пользоваться A100 40GB в Colab по 37 часов в месяц](https://zenn.dev/grand2/articles/ddba80ba400f6f)

### 💡 Особенности и преимущества
- **Оптимизация под A100 GPU**: Стабильная загрузка крупной модели MiniMax-H3 (`int8_convrot`) + Text Encoder + Audio/Video VAE без нехватки видеопамяти.
- **🚀 Ускорение с Turbo LoRA (рекомендуется)**: Автоматическая установка Turbo LoRA, которая сокращает генерацию со стандартных 25 шагов (~30 минут) до **6–8 шагов (~7 минут, в 4 раза быстрее)**.
- **Стек ускорения**: Автоматическая настройка современных нод ускорения `SageAttention` и `TeaCache` одним переключателем.
- **Полное сохранение данных на Google Drive (кэш моделей и готовые видео)**:
  - Модели и LoRA (~20 ГБ) кэшируются в `MyDrive/ComfyUI_Models/`, поэтому при повторных запусках скачивание пропускается (старт за 0 секунд).
  - Сгенерированные видео и изображения **автоматически сохраняются в `MyDrive/ComfyUI_Outputs/`**. Даже если среда выполнения отключится, результаты не пропадут.
- **Cloudflare Tunnel (бесплатно и без токенов)**: Автоматическая выдача защищённой публичной ссылки (`trycloudflare.com`) без необходимости регистрироваться в ngrok.

> ⚠️ **Внимание**: Убедитесь, что в верхнем меню ноутбука **«Среда выполнения» (Runtime) → «Сменить среду выполнения» (Change runtime type)** выбран ускоритель **GPU (A100)**.

## Step 1: Проверка окружения и подключение Google Drive (сохранение данных)

In [ ]:
# Проверка выделенного GPU и версии CUDA (убедитесь, что подключён A100)
!nvidia-smi

# Сохранение кэша моделей и готовых видео на Google Drive
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive успешно подключён!")

## Step 2: Установка ComfyUI и стека ускорения

- **ENABLE_ACCELERATION (рекомендуется)**: Включает ноды ускорения `SageAttention` и `TeaCache`.
- **Совместимость с CUDA 12.8 / 13.0**: Корректная работа PyTorch cu128 с драйвером хоста CUDA 13.0.

In [ ]:
import os

# @title Настройки ускорения
ENABLE_ACCELERATION = True  # @param {type:"boolean"}
INSTALL_TEACACHE = True     # @param {type:"boolean"}

%cd /content

# Клонирование актуальной версии ComfyUI
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# Установка зависимостей
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# Установка SageAttention и Triton
if ENABLE_ACCELERATION:
    print("⚡ Установка стека ускорения (SageAttention, Triton и др.)...")
    !pip install -q triton sageattention

# Установка ComfyUI-Manager
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# Ноды MiniMax-H3 Easy
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

# Ноды ускорения TeaCache
if INSTALL_TEACACHE and not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TeaCache"):
    !git clone https://github.com/chengzeyi/ComfyUI-MiniMaxH3-TeaCache.git || true

%cd /content/ComfyUI
print("✅ Все ноды и зависимости успешно установлены!")

## Step 3: Загрузка моделей MiniMax-H3 и Turbo LoRA (интеграция с Google Drive)

- **Turbo LoRA (`DOWNLOAD_TURBO_LORA = True`)**: Скачивает `lightx2v/Minimax-h3-Turbo` для **быстрой генерации за 8 шагов (в 4 раза быстрее)**.
- Если `USE_GOOGLE_DRIVE = True`:
  - **Кэш моделей и LoRA**: `/content/drive/MyDrive/ComfyUI_Models/` (повторное скачивание при следующих запусках не требуется)
  - **Папка для готовых видео**: `/content/drive/MyDrive/ComfyUI_Outputs/` (файлы сохраняются даже после удаления среды выполнения)

In [ ]:
import os
from huggingface_hub import hf_hub_download

# @title Настройка загрузки Turbo LoRA
DOWNLOAD_TURBO_LORA = True  # @param {type:"boolean"}

# Подготовка директорий для моделей и вывода
if USE_GOOGLE_DRIVE:
    MODELS_BASE = "/content/drive/MyDrive/ComfyUI_Models"
    OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI_Outputs"
    os.makedirs(MODELS_BASE, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Символическая ссылка папки вывода на Google Drive
    local_output = "/content/ComfyUI/output"
    if os.path.exists(local_output) and not os.path.islink(local_output):
        !rm -rf {local_output}
    if not os.path.exists(local_output):
        os.symlink(OUTPUT_DIR, local_output)

    # 2. Символические ссылки папок с моделями на Google Drive
    for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
        drive_sub = os.path.join(MODELS_BASE, sub)
        local_sub = os.path.join("/content/ComfyUI/models", sub)
        os.makedirs(drive_sub, exist_ok=True)
        if os.path.exists(local_sub) and not os.path.islink(local_sub):
            !rm -rf {local_sub}
        if not os.path.exists(local_sub):
            os.symlink(drive_sub, local_sub)
    print(f"📁 Кэш и папка вывода привязаны к Google Drive:\n  - Models: {MODELS_BASE}\n  - Outputs: {OUTPUT_DIR}")
else:
    MODELS_BASE = "/content/ComfyUI/models"

# 1. Базовые модели MiniMax-H3
REPO_ID = "Comfy-Org/MiniMax-H3"
files_to_download = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", REPO_ID),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", REPO_ID),
    ("vae/minimax_h3_video_vae_fp16.safetensors", REPO_ID),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", REPO_ID)
]

# 2. Turbo LoRA (8-шаговая дистилляция)
if DOWNLOAD_TURBO_LORA:
    files_to_download.append((
        "loras/minimax_h3_fl2v_turbo_8step_v1.0_768p_bf16.safetensors",
        "lightx2v/Minimax-h3-Turbo"
    ))

print("📥 Проверка и скачивание моделей MiniMax-H3 и Turbo LoRA...")
for filename, repo in files_to_download:
    target_path = os.path.join(MODELS_BASE, filename)
    sub_folder = os.path.dirname(filename)
    file_name_only = os.path.basename(filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1000000:
        print(f"⚡ Найдено в кэше (пропуск загрузки): {filename}")
    else:
        print(f"⬇️ Скачивание: {filename} (из {repo}) ...")
        hf_hub_download(
            repo_id=repo,
            filename=file_name_only if repo != REPO_ID else filename,
            local_dir=os.path.join(MODELS_BASE, sub_folder) if repo != REPO_ID else MODELS_BASE,
            local_dir_use_symlinks=False
        )

print("✅ Все модели и Turbo LoRA готовы к работе!")

## Step 4: Запуск ComfyUI и подключение через Cloudflare Tunnel

Запускает ComfyUI в фоновом режиме, создаёт защищённый публичный URL через Cloudflare Tunnel (`trycloudflare.com`) и **удерживает сервер в рабочем состоянии**.
Нажмите на появившуюся ссылку `https://xxxx.trycloudflare.com`, чтобы открыть веб-интерфейс ComfyUI (или передайте эту ссылку ИИ-агенту).

### 💡 Как собрать сверхбыстрый воркфлоу с Turbo LoRA
1. В интерфейсе ComfyUI сразу после загрузчика модели (`UNETLoader`) добавьте ноду **`LoraLoader`** (или `LoraLoaderModelOnly`).
2. Выберите в ней LoRA **`minimax_h3_fl2v_turbo_8step_...safetensors`**.
3. В ноде **KSampler уменьшите количество шагов (`steps`) с `25` до `8`** (рекомендуется `euler` / `normal`).
4. Теперь генерация одного качественного ролика займёт **всего 6–8 минут**!

In [ ]:
import subprocess
import threading
import time
import re

# Скачивание Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# Запуск ComfyUI в фоновом режиме
print("⚡ Запуск ComfyUI...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Вывод логов ComfyUI в отдельном потоке
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# Небольшая пауза и запуск туннеля Cloudflared
time.sleep(5)
print("🌐 Запуск Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not url_found:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        print("💡 Сервер запущен и работает. (Чтобы остановить его, нажмите кнопку Стоп у этой ячейки или отключите среду выполнения через верхнее меню)")
        url_found = True

# Удерживаем ячейку активной, пока работает процесс туннеля
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ Сервер остановлен.")
    comfy_proc.terminate()
    tunnel_proc.terminate()

## 🛑 (После завершения работы) Как гарантированно остановить списание CU

Когда вы закончите генерацию видео, **чтобы со 100% надёжностью остановить расход Compute Units (CU)**, удалите инстанс через меню:

### 💡 【Рекомендуется・100% надёжность】Отключение через меню Colab
1. Нажмите **«Среда выполнения» (Runtime)** в верхнем меню Colab.
2. Выберите **«Отключиться и удалить среду выполнения» (Disconnect and delete runtime)** и подтвердите действие.

> Все готовые видео в реальном времени сохраняются на Google Drive (`MyDrive/ComfyUI_Outputs/`), поэтому удалять среду выполнения полностью безопасно.